# ทดลอง/จูนการตัด background — TMD radar

notebook นี้เป็นแค่ **ตัวเรียกใช้** ตรรกะจริงอยู่ใน `radar_archive/*.py` ทั้งหมด
ใช้สำหรับดูผล จูน `lab_tolerance` และตรวจว่ามี overlay อะไรหลุดมาบ้าง


In [ ]:
# --- ติดตั้ง (รันบน Google Colab) ---
!git clone -q https://github.com/USERNAME/tmd-radar-archive.git
%cd tmd-radar-archive
!pip install -q -r requirements.txt
!apt-get install -y -qq tesseract-ocr > /dev/null

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from PIL import Image

from radar_archive.config import get_station
from radar_archive import fetch, palette, strip

st = get_station("PHS")
st

## 1. ดึงเฟรมล่าสุด (หรือใช้ไฟล์ที่มีอยู่)

In [ ]:
f = fetch.fetch_latest(st)          # ดึงสด
# f = fetch.fetch_from_file(st, "data/raw/PHS/2026/09/PHS_20260902_1145Z.jpg")   # หรือใช้ไฟล์เดิม

print("เวลาภาพ (UTC):", f.timestamp, "| ที่มาของเวลา:", f.timestamp_source)
print("เวลาไทย       :", fetch.th_time(f.timestamp))
f.image

## 2. palette ที่สกัดได้จาก colorbar

In [ ]:
pal_rgb, pal_dbz = palette.extract_palette(f.image, st)

fig, ax = plt.subplots(figsize=(9, 1.6))
ax.imshow(pal_rgb.reshape(1, -1, 3).astype(np.uint8) / 255, aspect="auto")
ax.set_xticks(range(len(pal_dbz)))
ax.set_xticklabels([f"{d:.0f}" for d in pal_dbz], fontsize=7)
ax.set_yticks([]); ax.set_xlabel("dBZ (โดยประมาณ)")
plt.show()

## 3. เทียบผลที่ tolerance ต่าง ๆ

- ต่ำไป → echo อ่อน ๆ หาย
- สูงไป → เส้นแม่น้ำ / range ring / ตัวอักษร เริ่มหลุดเข้ามา


In [ ]:
tols = [8, 10, 12, 15]
fig, ax = plt.subplots(1, len(tols) + 1, figsize=(4.2 * (len(tols) + 1), 4.6))
ax[0].imshow(np.asarray(strip.crop_plot(f.image, st)))
ax[0].set_title("original")
for a, tol in zip(ax[1:], tols):
    r = strip.strip_background(f.image, st, pal_rgb, tolerance=tol)
    a.imshow(np.asarray(strip.render_solid(r, (0, 0, 0))))
    a.set_title(f"tol={tol}  cov={r['coverage_pct']:.2f}%")
for a in ax: a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()

## 4. ดูว่า pixel ไหนถูกทิ้งเพราะอะไร

แผนที่ระยะสี (Lab distance) ช่วยหาว่ามี overlay ตัวไหนที่สีใกล้ palette จนเสี่ยงหลุดเข้ามา


In [ ]:
r = strip.strip_background(f.image, st, pal_rgb)
fig, ax = plt.subplots(1, 2, figsize=(13, 5.6))
im = ax[0].imshow(np.clip(r["distance"], 0, 40), cmap="magma_r")
ax[0].set_title("ระยะสีไปยัง palette ที่ใกล้ที่สุด (Lab)")
plt.colorbar(im, ax=ax[0], fraction=0.046)
ax[1].imshow(r["mask"], cmap="gray"); ax[1].set_title("echo mask")
for a in ax: a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()

## 5. ค่า dBZ เป็น array

In [ ]:
dbz = strip.to_dbz(r, pal_dbz)
print("shape:", dbz.shape, "| max:", np.nanmax(dbz), "| echo px:", int(r["mask"].sum()))

plt.figure(figsize=(7, 7))
plt.imshow(dbz, cmap="turbo", vmin=10, vmax=60)
plt.colorbar(label="dBZ"); plt.xticks([]); plt.yticks([])
plt.title("reflectivity ที่กู้กลับมาจากภาพ"); plt.show()